# 🗄️ SQLite DB Explorer

In [ ]:
# ─────────────────────────────────────────
#  CONFIG — edit only this cell
# ─────────────────────────────────────────
DB_PATH = "./foodguard.db"   # path to your SQLite file
ROWS_PER_PAGE = 50               # rows shown per table page
# ─────────────────────────────────────────

In [ ]:
import sqlite3, math, html
from IPython.display import display, HTML
import ipywidgets as widgets

# ── helpers ──────────────────────────────────────────────────────────────────

def get_conn():
    return sqlite3.connect(DB_PATH)

def get_tables():
    with get_conn() as c:
        rows = c.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()
    return [r[0] for r in rows]

def get_schema(table):
    with get_conn() as c:
        return c.execute(f"PRAGMA table_info('{table}')").fetchall()

def get_row_count(table):
    with get_conn() as c:
        return c.execute(f"SELECT COUNT(*) FROM \"{table}\"").fetchone()[0]

def get_rows(table, offset=0, limit=ROWS_PER_PAGE, search="", col=None):
    with get_conn() as c:
        cols = [r[1] for r in get_schema(table)]
        where = ""
        params = []
        if search and col:
            where = f'WHERE CAST(\"{col}\" AS TEXT) LIKE ?'
            params.append(f"%{search}%")
        elif search:
            clauses = [f'CAST(\"{c}\" AS TEXT) LIKE ?' for c in cols]
            where = "WHERE " + " OR ".join(clauses)
            params = [f"%{search}%"] * len(cols)
        total = c.execute(f'SELECT COUNT(*) FROM \"{table}\" {where}', params).fetchone()[0]
        rows  = c.execute(f'SELECT * FROM \"{table}\" {where} LIMIT ? OFFSET ?', params + [limit, offset]).fetchall()
    return cols, rows, total

def run_query(sql):
    with get_conn() as c:
        cur = c.execute(sql)
        cols = [d[0] for d in (cur.description or [])]
        rows = cur.fetchall() if cols else []
    return cols, rows

# ── CSS ──────────────────────────────────────────────────────────────────────

CSS = """
<style>
:root {
  --bg:        #0f1117;
  --surface:   #1a1d27;
  --border:    #2a2d3e;
  --accent:    #4f8ef7;
  --accent2:   #7c5cbf;
  --text:      #e2e4ed;
  --muted:     #6b7280;
  --danger:    #ef4444;
  --success:   #22c55e;
  --mono:      'JetBrains Mono', 'Fira Code', monospace;
}
#dbe-root * { box-sizing: border-box; font-family: 'Inter', system-ui, sans-serif; }
#dbe-root {
  display: flex; gap: 0; border: 1px solid var(--border);
  border-radius: 10px; overflow: hidden;
  background: var(--bg); color: var(--text);
  min-height: 520px; font-size: 13px;
}
/* sidebar */
#dbe-sidebar {
  width: 210px; min-width: 210px; background: var(--surface);
  border-right: 1px solid var(--border); display: flex; flex-direction: column;
}
#dbe-sidebar-header {
  padding: 14px 14px 10px; font-size: 11px; font-weight: 700;
  letter-spacing: .08em; text-transform: uppercase; color: var(--muted);
  border-bottom: 1px solid var(--border);
}
#dbe-tables { flex: 1; overflow-y: auto; padding: 6px 0; }
.dbe-table-item {
  padding: 8px 16px; cursor: pointer; display: flex;
  align-items: center; gap: 8px; border-left: 3px solid transparent;
  transition: background .15s;
}
.dbe-table-item:hover  { background: rgba(79,142,247,.08); }
.dbe-table-item.active { background: rgba(79,142,247,.13); border-left-color: var(--accent); color: var(--accent); }
.dbe-table-icon { font-size: 12px; opacity: .6; }
.dbe-table-count { margin-left: auto; font-size: 10px; color: var(--muted); }
/* main */
#dbe-main { flex: 1; display: flex; flex-direction: column; overflow: hidden; }
#dbe-toolbar {
  padding: 10px 16px; border-bottom: 1px solid var(--border);
  display: flex; align-items: center; gap: 10px; background: var(--surface);
}
#dbe-toolbar input {
  flex: 1; padding: 6px 10px; border-radius: 6px;
  border: 1px solid var(--border); background: var(--bg);
  color: var(--text); font-family: var(--mono); font-size: 12px; outline: none;
}
#dbe-toolbar input:focus { border-color: var(--accent); }
.dbe-btn {
  padding: 6px 12px; border-radius: 6px; border: none; cursor: pointer;
  font-size: 12px; font-weight: 600; transition: opacity .15s;
}
.dbe-btn:hover { opacity: .85; }
.dbe-btn-primary { background: var(--accent); color: #fff; }
.dbe-btn-ghost   { background: var(--border); color: var(--text); }
/* tabs */
#dbe-tabs {
  display: flex; border-bottom: 1px solid var(--border);
  background: var(--surface); padding: 0 16px;
}
.dbe-tab {
  padding: 9px 14px; cursor: pointer; font-size: 12px; font-weight: 600;
  color: var(--muted); border-bottom: 2px solid transparent; margin-bottom: -1px;
  transition: color .15s;
}
.dbe-tab:hover  { color: var(--text); }
.dbe-tab.active { color: var(--accent); border-bottom-color: var(--accent); }
/* panels */
#dbe-content { flex: 1; overflow: auto; padding: 0; }
.dbe-panel { display: none; height: 100%; }
.dbe-panel.active { display: block; }
/* data table */
.dbe-grid { width: 100%; border-collapse: collapse; font-family: var(--mono); font-size: 12px; }
.dbe-grid thead th {
  position: sticky; top: 0; background: var(--surface); padding: 9px 12px;
  text-align: left; border-bottom: 1px solid var(--border);
  color: var(--muted); font-weight: 700; font-size: 11px;
  letter-spacing: .05em; white-space: nowrap;
}
.dbe-grid tbody td {
  padding: 7px 12px; border-bottom: 1px solid rgba(42,45,62,.6);
  max-width: 300px; overflow: hidden; text-overflow: ellipsis; white-space: nowrap;
  vertical-align: top;
}
.dbe-grid tbody tr:hover td { background: rgba(79,142,247,.05); }
.dbe-null { color: var(--muted); font-style: italic; }
.dbe-num  { color: #a78bfa; }
.dbe-bool { color: var(--success); }
/* pagination */
#dbe-pagination {
  padding: 10px 16px; border-top: 1px solid var(--border);
  display: flex; align-items: center; gap: 8px;
  background: var(--surface); font-size: 12px; color: var(--muted);
}
#dbe-pagination span { margin: 0 auto; }
/* schema */
.dbe-schema-table { width: 100%; border-collapse: collapse; font-size: 12px; }
.dbe-schema-table th {
  padding: 8px 14px; text-align: left;
  border-bottom: 1px solid var(--border);
  color: var(--muted); font-size: 11px; font-weight: 700;
  background: var(--surface);
}
.dbe-schema-table td {
  padding: 7px 14px; border-bottom: 1px solid rgba(42,45,62,.5);
  font-family: var(--mono);
}
.dbe-tag {
  display: inline-block; padding: 1px 7px; border-radius: 4px;
  font-size: 10px; font-weight: 700;
}
.dbe-tag-pk  { background: rgba(79,142,247,.18); color: var(--accent); }
.dbe-tag-nn  { background: rgba(239,68,68,.14);  color: var(--danger); }
.dbe-type    { color: #f97316; }
/* query */
#dbe-query-panel { padding: 14px 16px; display: flex; flex-direction: column; gap: 10px; }
#dbe-sql-input {
  width: 100%; height: 110px; padding: 10px 12px;
  border-radius: 7px; border: 1px solid var(--border);
  background: #0b0d14; color: var(--text);
  font-family: var(--mono); font-size: 12px;
  resize: vertical; outline: none;
}
#dbe-sql-input:focus { border-color: var(--accent); }
#dbe-query-result { overflow: auto; }
.dbe-error { color: var(--danger); font-family: var(--mono); font-size: 12px; padding: 8px; }
.dbe-ok    { color: var(--success); font-size: 12px; padding: 8px; }
/* welcome */
#dbe-welcome {
  display: flex; flex-direction: column; align-items: center; justify-content: center;
  height: 100%; gap: 10px; color: var(--muted); font-size: 13px;
}
#dbe-welcome h2 { font-size: 18px; color: var(--text); margin: 0; }
</style>
"""

# ── HTML builder ─────────────────────────────────────────────────────────────

def _cell(val):
    if val is None:
        return '<td><span class="dbe-null">NULL</span></td>'
    if isinstance(val, (int, float)):
        return f'<td class="dbe-num">{html.escape(str(val))}</td>'
    return f'<td title="{html.escape(str(val), quote=True)}">{html.escape(str(val))}</td>'

def render_data_table(cols, rows, total, page, n_pages):
    thead = "".join(f"<th>{html.escape(c)}</th>" for c in cols)
    tbody = "".join(
        "<tr>" + "".join(_cell(v) for v in row) + "</tr>"
        for row in rows
    )
    if not rows:
        tbody = f'<tr><td colspan="{len(cols)}" style="text-align:center;color:var(--muted);padding:24px">No rows found</td></tr>'
    return f"""
    <table class="dbe-grid">
      <thead><tr>{thead}</tr></thead>
      <tbody>{tbody}</tbody>
    </table>
    """, f"Page {page} of {max(n_pages,1)} &nbsp;·&nbsp; {total} row{'s' if total!=1 else ''}"

def render_schema(table):
    rows = get_schema(table)
    trs = ""
    for r in rows:
        cid, name, typ, notnull, dflt, pk = r
        tags = ""
        if pk:      tags += '<span class="dbe-tag dbe-tag-pk">PK</span> '
        if notnull: tags += '<span class="dbe-tag dbe-tag-nn">NOT NULL</span>'
        dflt_str = html.escape(str(dflt)) if dflt is not None else '<span class="dbe-null">—</span>'
        trs += f"""
        <tr>
          <td>{cid}</td>
          <td><b>{html.escape(name)}</b></td>
          <td class="dbe-type">{html.escape(typ or '—')}</td>
          <td>{dflt_str}</td>
          <td>{tags or '<span class="dbe-null">—</span>'}</td>
        </tr>"""
    return f"""
    <table class="dbe-schema-table">
      <thead><tr><th>#</th><th>Column</th><th>Type</th><th>Default</th><th>Flags</th></tr></thead>
      <tbody>{trs}</tbody>
    </table>"""

# ── state ─────────────────────────────────────────────────────────────────────

state = dict(table=None, page=1, search="", tab="data", query_sql="")

# ── widget skeleton ───────────────────────────────────────────────────────────

out = widgets.Output()

def build_sidebar(tables, counts):
    items = ""
    for t in tables:
        active = "active" if t == state["table"] else ""
        items += f"""
        <div class="dbe-table-item {active}" onclick="dbeSelectTable('{t}')">
          <span class="dbe-table-icon">▦</span>
          <span>{html.escape(t)}</span>
          <span class="dbe-table-count">{counts.get(t, '?')}</span>
        </div>"""
    return f"""
    <div id="dbe-sidebar">
      <div id="dbe-sidebar-header">Tables ({len(tables)})</div>
      <div id="dbe-tables">{items}</div>
    </div>"""

def build_main():
    t = state["table"]
    if not t:
        return """
        <div id="dbe-main">
          <div id="dbe-welcome">
            <h2>SQLite Explorer</h2>
            <p>Select a table from the sidebar to browse data.</p>
          </div>
        </div>"""

    # data tab
    offset = (state["page"] - 1) * ROWS_PER_PAGE
    cols, rows, total = get_rows(t, offset=offset, search=state["search"])
    n_pages = max(1, math.ceil(total / ROWS_PER_PAGE))
    table_html, pager_info = render_data_table(cols, rows, total, state["page"], n_pages)

    can_prev = state["page"] > 1
    can_next = state["page"] < n_pages

    data_tab_active    = "active" if state["tab"] == "data"   else ""
    schema_tab_active  = "active" if state["tab"] == "schema" else ""
    query_tab_active   = "active" if state["tab"] == "query"  else ""

    data_panel_active   = "active" if state["tab"] == "data"   else ""
    schema_panel_active = "active" if state["tab"] == "schema" else ""
    query_panel_active  = "active" if state["tab"] == "query"  else ""

    schema_html = render_schema(t)

    # query tab
    escaped_sql = html.escape(state.get("query_sql") or f'SELECT * FROM "{t}" LIMIT 100;', quote=True)
    query_result_html = ""
    if state.get("query_result"):
        r = state["query_result"]
        if "error" in r:
            query_result_html = f'<div class="dbe-error">⚠ {html.escape(r["error"])}</div>'
        elif r["cols"]:
            qh, _ = render_data_table(r["cols"], r["rows"], len(r["rows"]), 1, 1)
            query_result_html = qh
        else:
            query_result_html = f'<div class="dbe-ok">✓ Query executed. {r.get("msg","")}</div>'

    prev_disabled = "" if can_prev else "disabled style='opacity:.4;cursor:default'"
    next_disabled = "" if can_next else "disabled style='opacity:.4;cursor:default'"

    return f"""
    <div id="dbe-main">
      <div id="dbe-toolbar">
        <b style="font-size:13px;color:var(--accent)">{html.escape(t)}</b>
        <input id="dbe-search" type="text" placeholder="Search all columns…"
               value="{html.escape(state['search'], quote=True)}"
               oninput="dbeSearch(this.value)"
               style="max-width:260px" />
        <button class="dbe-btn dbe-btn-ghost" onclick="dbeCopyDDL()">Copy DDL</button>
      </div>
      <div id="dbe-tabs">
        <div class="dbe-tab {data_tab_active}"   onclick="dbeTab('data')">Data</div>
        <div class="dbe-tab {schema_tab_active}" onclick="dbeTab('schema')">Structure</div>
        <div class="dbe-tab {query_tab_active}"  onclick="dbeTab('query')">SQL</div>
      </div>
      <div id="dbe-content">
        <div class="dbe-panel {data_panel_active}" id="dbe-panel-data">
          {table_html}
        </div>
        <div class="dbe-panel {schema_panel_active}" id="dbe-panel-schema">
          {schema_html}
        </div>
        <div class="dbe-panel {query_panel_active}" id="dbe-panel-query">
          <div id="dbe-query-panel">
            <textarea id="dbe-sql-input" placeholder="Enter SQL…">{escaped_sql}</textarea>
            <div><button class="dbe-btn dbe-btn-primary" onclick="dbeRunQuery()">▶ Run Query</button></div>
            <div id="dbe-query-result">{query_result_html}</div>
          </div>
        </div>
      </div>
      <div id="dbe-pagination">
        <button class="dbe-btn dbe-btn-ghost" onclick="dbePrev()" {prev_disabled}>← Prev</button>
        <span>{pager_info}</span>
        <button class="dbe-btn dbe-btn-ghost" onclick="dbeNext()" {next_disabled}>Next →</button>
      </div>
    </div>"""

def build_js(tables):
    tables_js = str(tables)
    return f"""
    <script>
    // debounce
    let _dbeST = null;
    function dbeSearch(v){{
      clearTimeout(_dbeST);
      _dbeST = setTimeout(() => {{
        var k = IPython.notebook.kernel;
        k.execute("state['search']=r\\\"\\\"\\\"" + v + "\\\"\\\"\\\"; state['page']=1; _render()");
      }}, 300);
    }}
    function dbeSelectTable(t){{
      var k = IPython.notebook.kernel;
      k.execute("state['table']='" + t + "'; state['page']=1; state['search']=''; state.pop('query_result',None); _render()");
    }}
    function dbeTab(t){{
      var k = IPython.notebook.kernel;
      k.execute("state['tab']='" + t + "'; _render()");
    }}
    function dbePrev(){{
      var k = IPython.notebook.kernel;
      k.execute("state['page']=max(1,state['page']-1); _render()");
    }}
    function dbeNext(){{
      var k = IPython.notebook.kernel;
      k.execute("state['page']+=1; _render()");
    }}
    function dbeRunQuery(){{
      var sql = document.getElementById('dbe-sql-input').value;
      var escaped = sql.replace(/\\/g,'\\\\').replace(/'/g,"\\'");
      var k = IPython.notebook.kernel;
      k.execute("state['query_sql']=r'" + escaped + "'; _run_query(); _render()");
    }}
    function dbeCopyDDL(){{
      var k = IPython.notebook.kernel;
      k.execute("_copy_ddl('" + {repr(state.get('table',''))} + "')");
    }}
    </script>
    """

def _render():
    tables = get_tables()
    counts = {}
    for t in tables:
        try: counts[t] = get_row_count(t)
        except: counts[t] = "?"
    sidebar = build_sidebar(tables, counts)
    main    = build_main()
    js      = build_js(tables)
    full = CSS + f'<div id="dbe-root">{sidebar}{main}</div>' + js
    out.clear_output(wait=True)
    with out:
        display(HTML(full))

def _run_query():
    sql = state.get("query_sql", "")
    try:
        cols, rows = run_query(sql)
        if cols:
            state["query_result"] = {"cols": cols, "rows": rows}
        else:
            state["query_result"] = {"cols": [], "rows": [], "msg": "Done."}
    except Exception as e:
        state["query_result"] = {"error": str(e)}

def _copy_ddl(table):
    try:
        with get_conn() as c:
            ddl = c.execute(f"SELECT sql FROM sqlite_master WHERE name=?", (table,)).fetchone()
        print(ddl[0] if ddl else "Not found")
    except Exception as e:
        print(f"Error: {e}")

# ── launch ────────────────────────────────────────────────────────────────────
display(out)
_render()
print(f"Connected to: {DB_PATH}")